In [ ]:
import pandas as pd
from pathlib import Path
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score, recall_score
import joblib

X_train = pd.read_parquet("../data/X_train.parquet")
y_train = pd.read_parquet("../data/y_train.parquet").values.ravel()
X_val = pd.read_parquet("../data/X_val.parquet")
y_val = pd.read_parquet("../data/y_val.parquet").values.ravel()

In [7]:
import gc
X_train = X_train.astype('float32')
X_val = X_val.astype('float32')

def evaluate_method(name, X_resampled, y_resampled, X_v, y_v):
    model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    model.fit(X_resampled, y_resampled)
    
    y_pred = model.predict(X_v)
    y_proba = model.predict_proba(X_v)[:, 1]
    
    print(f"\n[PHƯƠNG PHÁP: {name}]")
    print(classification_report(y_v, y_pred))
    print(f"AUPRC: {average_precision_score(y_v, y_proba):.4f}")
    return recall_score(y_v, y_pred), average_precision_score(y_v, y_proba)

smote = SMOTE(sampling_strategy=0.3, random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
rec_smote, au_smote = evaluate_method("SMOTE (Ratio 0.3)", X_smote, y_smote, X_val, y_val)

del X_smote, y_smote
gc.collect() 

rus = RandomUnderSampler(sampling_strategy=0.3, random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
rec_rus, au_rus = evaluate_method("Random Under-sampling (Ratio 0.5)", X_rus, y_rus, X_val, y_val)

print("\n" + "="*30)
print(f"SMOTE AUPRC: {au_smote:.4f}")
print(f"RUS AUPRC:   {au_rus:.4f}")
print("="*30)


[PHƯƠNG PHÁP: SMOTE (Ratio 0.3)]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    114044
           1       0.81      0.30      0.44      4064

    accuracy                           0.97    118108
   macro avg       0.89      0.65      0.71    118108
weighted avg       0.97      0.97      0.97    118108

AUPRC: 0.4561

[PHƯƠNG PHÁP: Random Under-sampling (Ratio 0.5)]
              precision    recall  f1-score   support

           0       0.98      0.98      0.98    114044
           1       0.42      0.50      0.45      4064

    accuracy                           0.96    118108
   macro avg       0.70      0.74      0.72    118108
weighted avg       0.96      0.96      0.96    118108

AUPRC: 0.4606

SMOTE AUPRC: 0.4561
RUS AUPRC:   0.4606


Sử dụng RUS (Random Under-Sampling) để xử lý mất cân bằng dữ liệu vì có thể nhận biết được khoảng 50% các giao dịch gian lận, cao hơn Smote chỉ 30%, chỉ số AUPRC của RUS cũng cao hơn

In [ ]:
counts = pd.Series(y_rus).value_counts()
percent = pd.Series(y_rus).value_counts(normalize=True) * 100
print(f"\n--- Thống kê tập Train SAU KHI RUS ---")
print(f"Lớp 0 (Non-Fraud): {counts[0]} mẫu ({percent[0]:.2f}%)")
print(f"Lớp 1 (Fraud):     {counts[1]} mẫu ({percent[1]:.2f}%)")

# X_rus.to_csv("../data/X_train_rus.parquet", index=False)
# y_rus_df = pd.DataFrame(y_rus, columns=['isFraud'])
# y_rus_df.to_csv("../data/y_train_rus.parquet", index=False)


--- Thống kê tập Train SAU KHI RUS ---
Lớp 0 (Non-Fraud): 55330 mẫu (76.92%)
Lớp 1 (Fraud):     16599 mẫu (23.08%)
